Original repository available at:
https://github.com/naoto0804/pytorch-inpainting-with-partial-conv

In [ ]:
import os

# Zipped repository
zip_file_path = '/content/pytorch-inpainting-with-partial-conv-0.0.0.zip'

# Check if the zip file exists before attempting to unzip
if os.path.exists(zip_file_path):
    # Unzip the file to the current directory
    os.system(f'unzip -q {zip_file_path} -d /content/')
    print(f'Successfully unzipped {zip_file_path} to /content/')
    # List the contents of the directory where the files were unzipped
    print('\nContents after unzipping:')
    os.system('ls -F /content/')
else:
    print(f'Error: The file {zip_file_path} does not exist.')

In [ ]:
# os.system(f'unzip -q /content/PlacesTestSet.zip -d /content/')
# os.system(f'unzip -q /content/masks_variety_for_iizuka.zip -d /content/')

In [ ]:
# Install dependencies from requirements.txt
# Due to issues with old versions, install numpy and Pillow without strict version constraints first.
%pip install numpy Pillow --upgrade --user
%pip install -r /content/pytorch-inpainting-with-partial-conv/requirements.txt

In [ ]:
import sys
# Add the project directory to the Python path to import modules
sys.path.append('/content/pytorch-inpainting-with-partial-conv-0.0.0')

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import torchvision.utils as vutils

# Net structure for model loading
from net import PConvUNet 
from places2 import Places2

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# We initialize the generator model based on the structure in "net.py"
generator = PConvUNet(layer_size=7).to(device)

# Load pre-trained weights
model_path = '/content/iter_1000000.pth'
if torch.cuda.is_available():
    checkpoint = torch.load(model_path)
else:
    checkpoint = torch.load(model_path, map_location=torch.device('cpu'))

# Load the state dictionary into the model
if isinstance(checkpoint, dict) and 'model' in checkpoint:
    generator.load_state_dict(checkpoint['model'])
elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    generator.load_state_dict(checkpoint['state_dict'])
else:
    generator.load_state_dict(checkpoint)

generator.eval() # Set model to evaluation mode
print("Generator loaded successfully.")

In [ ]:
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# Ensure opt module is imported for MEAN and STD
import opt
from util.image import unnormalize # Import the unnormalize utility function

# The generator model and device should already be defined from previous cells
# generator, device, model_path, image_size are available

image_size = 256

size = (image_size, image_size)
img_transform = transforms.Compose(
    [transforms.Resize(size=size), transforms.ToTensor(),
     transforms.Normalize(mean=opt.MEAN, std=opt.STD)])
mask_transform = transforms.Compose(
    [transforms.Resize(size=size), transforms.ToTensor()])

image_list = sorted([f for f in os.listdir("/content/PlacesTestSet")
                     if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])

mask_list = sorted([f for f in os.listdir("/content/masks_variety_for_iizuka")
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])

output_dir = "/content/PartConvOutput"
os.makedirs(output_dir, exist_ok=True)

counter = 0;

# Set model in evaluation mode
generator.eval() 

# In order to test the net for our configuration of 1 image with each mask, we replicate the structure
# of the test.py script.
for img_name in image_list:
  img_path = os.path.join("/content/PlacesTestSet", img_name)
  # Load image and mask
  img = Image.open(img_path).convert('RGB')
  # Apply transforms
  input_img_tensor = img_transform(img).unsqueeze(0).to(device) # Add batch dimension

  for mask_name in mask_list:
    mask_path = os.path.join("/content/masks_variety_for_iizuka", mask_name)
    mask = Image.open(mask_path).convert('RGB')
    input_mask_tensor = mask_transform(mask).unsqueeze(0).to(device)

    with torch.no_grad():
        output_tensor, _ = generator(input_img_tensor, input_mask_tensor)

    # Denormalize the model output tensor for display
    output_denorm = unnormalize(output_tensor.cpu()).squeeze(0)
    output_denorm = output_denorm.clamp(0, 1)

    # Convert tensors back to PIL Image for saving
    output_display = transforms.ToPILImage()(output_denorm)

    img_base = os.path.splitext(img_name)[0]
    mask_base = os.path.splitext(mask_name)[0]

    out_name = "{}__{}.png".format(img_base, mask_base)
    out_path = os.path.join(output_dir, out_name)
    output_display.save(out_path)

    counter += 1
    if (counter % 1000 == 0):
      print(f"Processed {counter} images.")

In [ ]:
output_list = sorted([f for f in os.listdir("/content/PartConvOutput")
                     if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])
output_test = output_list[:10]
for out in output_test:
  out_path = os.path.join("/content/PartConvOutput", out)
  img = Image.open(out_path).convert('RGB')
  plt.imshow(img)
  plt.show()

In [ ]:
# Download the output directory to save the data
!zip -r /content/PartConvOutput.zip /content/PartConvOutput

from google.colab import files
files.download('/content/PartConvOutput.zip')

In [ ]:
# Running our metrics (definitions in Metrics.ipynb)
pred_path = "/content/PartConvOutput/"
gt_path = "/content/PlacesTestSet/"
mask_path = "/content/masks_variety_for_iizuka/"

start_time = time.perf_counter()

metrics_array, mean_metrics = evaluate_folders(
    pred_dir=pred_path,
    gt_dir=gt_path,
    mask_input=48,
    metrics=("L1",  "PSNR", "SSIM"    , "LPIPS"),
    ssim_fn=ssim_fn
)

final_time = time.perf_counter() - start_time
print(f"Elapsed time: {final_time // 60:.0f}m {final_time % 60:.1f}s")

In [ ]:
# Saving results
np.save("metrics_partial_conv_places.npy", metrics_array)
np.save("mean_metrics_partial_conv_places.npy", mean_metrics)